# MCP-Based Distributed RAG System

This notebook demonstrates a **distributed Retrieval-Augmented Generation** system built on the
**Model Context Protocol (MCP)** with **LangGraph** agentic orchestration.

The system queries three heterogeneous knowledge sources in parallel:
1. **Vector Store** (semantic search over text chunks)
2. **Knowledge Graph** (entity-relationship traversal)
3. **Structured Database** (SQL-based technique/paper lookup)

Each source runs as an independent MCP server. The LangGraph orchestrator classifies queries,
plans retrieval routes, fuses results via Reciprocal Rank Fusion, evaluates quality,
and generates answers with source citations.

See [mcp-distributed-rag-architecture.md](mcp-distributed-rag-architecture.md) for the full
architectural design document with UML diagrams.


## 1. Installation


In [ ]:
%pip install --quiet --upgrade mcp[cli] langchain langchain-openai langgraph openai numpy python-dotenv

## 2. Environment Setup


In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")


## 3. Path Setup

Ensure the notebook can import from the local modules.


In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

print(f"Working directory: {notebook_dir}")
print(f"Python: {sys.executable}")


## 4. Inspect the MCP Servers

Before running the full orchestrator, let's verify that each MCP server starts
correctly and discover its available tools.


In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def inspect_server(name: str, script: str):
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[script],
        env={**os.environ},
    )
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_resp = await session.list_tools()
            print(f"\n=== {name} ===")
            for tool in tools_resp.tools:
                print(f"  Tool: {tool.name}")
                print(f"    {tool.description[:100]}")

servers = {
    "Vector Store": str(notebook_dir / "vector_store_server.py"),
    "Knowledge Graph": str(notebook_dir / "knowledge_graph_server.py"),
    "Structured DB": str(notebook_dir / "structured_db_server.py"),
}

for name, script in servers.items():
    await inspect_server(name, script)


## 5. Test Individual MCP Servers

Let's test each server independently to see what kind of results they return.


### 5.1 Vector Store — Semantic Search


In [ ]:
async def test_vector_store():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(notebook_dir / "vector_store_server.py")],
        env={**os.environ},
    )
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(
                "semantic_search",
                {"query": "What is task decomposition?", "top_k": 3},
            )
            import json
            data = json.loads(result.content[0].text)
            for item in data:
                print(f"[{item['score']:.3f}] {item['id']}: {item['text'][:120]}...")

await test_vector_store()


### 5.2 Knowledge Graph — Entity Traversal


In [ ]:
async def test_knowledge_graph():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(notebook_dir / "knowledge_graph_server.py")],
        env={**os.environ},
    )
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(
                "graph_query",
                {"entity": "Task Decomposition", "depth": 2},
            )
            import json
            data = json.loads(result.content[0].text)
            print(f"Entity: {data['entity']}, Depth: {data['depth']}")
            for triple in data['triples']:
                print(f"  {triple['subject']} --[{triple['predicate']}]--> {triple['object']}")

await test_knowledge_graph()


### 5.3 Structured DB — Technique Search


In [ ]:
async def test_structured_db():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(notebook_dir / "structured_db_server.py")],
        env={**os.environ},
    )
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(
                "search_techniques",
                {"keyword": "tool", "year_from": 2023},
            )
            import json
            data = json.loads(result.content[0].text)
            for row in data:
                print(f"  {row['name']} ({row['year']}) - {row['description'][:100]}...")

await test_structured_db()


## 6. Build & Run the Full Orchestrator

Now let's wire up the complete LangGraph orchestration pipeline that:
1. Classifies the query (LLM-based)
2. Plans which MCP servers to query
3. Fans out retrieval in parallel
4. Fuses results with Reciprocal Rank Fusion
5. Evaluates context quality
6. Reformulates if needed (iterative refinement)
7. Generates a cited answer


### 6.1 Initialize the Orchestrator


In [ ]:
from orchestrator import create_orchestrator

graph, pool = await create_orchestrator()
print("Orchestrator ready. MCP servers connected.")


### 6.2 Visualize the Graph

The LangGraph state machine with the conditional quality gate and reformulation loop.


In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))


### 6.3 Query 1 — Factual (Single Source)

A straightforward factual query that should primarily route to the vector store.


In [ ]:
result = await graph.ainvoke({"query": "What is task decomposition in LLM agents?"})

print("=== Classification ===")
print(f"  Type: {result['classification']['query_type']}")
print(f"  Domains: {result['classification']['domains']}")
print(f"  Entities: {result['classification']['extracted_entities']}")

print(f"\n=== Quality ===")
print(f"  Sufficient: {result['quality_assessment']['sufficient']}")
print(f"  Relevance: {result['quality_assessment']['relevance_score']:.3f}")
print(f"  Iterations: {result['iteration_count']}")

print(f"\n=== Fusion ===")
print(f"  Sources: {result['fused_context']['source_distribution']}")
print(f"  Total items: {len(result['fused_context']['ranked_items'])}")

print(f"\n=== Answer ===")
print(result["final_response"])


### 6.4 Query 2 — Relationship (Multi-Source)

A relationship query that should route to both the knowledge graph and vector store.


In [ ]:
result = await graph.ainvoke({"query": "How does ReAct relate to Chain of Thought and self-reflection?"})

print("=== Classification ===")
print(f"  Type: {result['classification']['query_type']}")
print(f"  Domains: {result['classification']['domains']}")
print(f"  Multi-hop: {result['classification']['requires_multi_hop']}")

print(f"\n=== Fusion ===")
print(f"  Sources: {result['fused_context']['source_distribution']}")

print(f"\n=== Answer ===")
print(result["final_response"])


### 6.5 Query 3 — Structured (Papers & Techniques)

A structured query about papers and techniques that should route to the structured DB.


In [ ]:
result = await graph.ainvoke({"query": "What tool-use techniques were published in 2023?"})

print("=== Classification ===")
print(f"  Type: {result['classification']['query_type']}")
print(f"  Domains: {result['classification']['domains']}")

print(f"\n=== Fusion ===")
print(f"  Sources: {result['fused_context']['source_distribution']}")

print(f"\n=== Answer ===")
print(result["final_response"])


### 6.6 Query 4 — Complex Multi-Source

A complex query that requires information from all three sources.


In [ ]:
result = await graph.ainvoke({
    "query": "Compare the memory architectures used in LLM agents and list the relevant papers"
})

print("=== Classification ===")
print(f"  Type: {result['classification']['query_type']}")
print(f"  Domains: {result['classification']['domains']}")
print(f"  Complexity: {result['classification']['complexity']}")

print(f"\n=== Quality ===")
print(f"  Sufficient: {result['quality_assessment']['sufficient']}")
print(f"  Relevance: {result['quality_assessment']['relevance_score']:.3f}")
print(f"  Iterations: {result['iteration_count']}")

print(f"\n=== Fusion ===")
print(f"  Sources: {result['fused_context']['source_distribution']}")
print(f"  Total items: {len(result['fused_context']['ranked_items'])}")

print(f"\n=== Answer ===")
print(result["final_response"])


### 6.7 Streaming Execution

Watch the orchestrator execute step by step using LangGraph's streaming mode.


In [ ]:
async for step in graph.astream(
    {"query": "What autonomous agent projects exist and how do they use planning?"},
    stream_mode="updates",
):
    for node_name, node_output in step.items():
        print(f"\n--- {node_name} ---")
        if node_name == "classify":
            c = node_output["classification"]
            print(f"  Type: {c['query_type']}, Domains: {c['domains']}")
        elif node_name == "plan":
            p = node_output["execution_plan"]
            for group in p["parallel_groups"]:
                calls = [(c["server_name"], c["tool_name"]) for c in group]
                print(f"  Parallel calls: {calls}")
        elif node_name == "retrieve":
            for r in node_output.get("retrieval_results", []):
                print(f"  {r['source']}: {len(r['items'])} items ({r['latency_ms']:.0f}ms)")
        elif node_name == "fuse":
            f = node_output["fused_context"]
            print(f"  Score: {f['overall_score']:.4f}, Sources: {f['source_distribution']}")
        elif node_name == "quality_check":
            q = node_output["quality_assessment"]
            print(f"  Sufficient: {q['sufficient']}, Relevance: {q['relevance_score']:.3f}")
        elif node_name == "reformulate":
            print(f"  Reformulated: {node_output['query']}")
        elif node_name == "generate":
            print(f"  Response length: {len(node_output['final_response'])} chars")
            print(f"\n{node_output['final_response']}")


## 7. Cleanup

Shut down the MCP server connections.


In [ ]:
await pool.close()
print("MCP connections closed.")


## 8. Architecture Summary

This demo implements the architecture described in
[mcp-distributed-rag-architecture.md](mcp-distributed-rag-architecture.md):

| Component | Implementation |
|-----------|---------------|
| **MCP Servers** | 3 FastMCP servers (vector store, knowledge graph, structured DB) |
| **Query Classifier** | GPT-4o-mini based intent detection |
| **Route Planner** | Domain-aware parallel execution planning |
| **Fan-Out Controller** |  for parallel MCP tool calls |
| **Result Fusion** | Reciprocal Rank Fusion (RRF) merging heterogeneous results |
| **Quality Gate** | Embedding-based cosine similarity evaluation |
| **Query Reformulator** | LLM-based query rewriting with partial context |
| **Generator** | GPT-4o-mini with multi-source context and citations |
| **Orchestrator** | LangGraph StateGraph with conditional edges |

### Key files

-  — MCP server wrapping OpenAI embeddings + cosine similarity search
-  — MCP server with in-memory entity-relationship graph
-  — MCP server wrapping SQLite for technique/paper queries
-  — LangGraph graph builder with classify → plan → retrieve → fuse → quality → generate pipeline
-  — Reciprocal Rank Fusion and weighted fusion implementations
-  — Embedding-based relevance scoring
-  — Shared data models (state, classifications, results)
-  — Pre-built sample data about LLM agents
